# Nimbus prediction notebook 

In [1]:
# import required packages
import warnings
warnings.simplefilter("ignore")
import os
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from nimbus_inference.nimbus import Nimbus, prep_naming_convention
from nimbus_inference.utils import MultiplexDataset
from alpineer import io_utils
from nimbus_inference import example_dataset
from nimbus_inference.viewer_widget import NimbusViewer

## 0: Set root directory and download example dataset
Here we are using the example data located in `/data/example_dataset/input_data`. To modify this notebook to run using your own data, simply change `base_dir` to point to your own sub-directory within the data folder. Set `base_dir`, the path to all of your imaging data (i.e. multiplexed images and segmentation masks). Subdirectory `nimbus_output` will contain all of the data generated by this notebook. In the following, we expect this folder structure, with `fov_1` and `fov_2` either being folders of individual channel images or `.ome.tiff` files that contain all channels in a single file.
```bash
|-- base_dir
|   |-- image_data
|   |   |-- fov_1
|   |   |-- fov_2
|   |-- segmentation
|   |   |-- deepcell_output
|   |-- nimbus_output
```

In [2]:
# set up the base directory
base_dir = os.path.normpath("/data1/lowes/ratnayn/Data/CellDive_analysis_data")

If you would like to test Nimbus with an example dataset, run the cell below. It will download a dataset consisting of 10 FOVs with 22 channels. You may find more information about the example dataset in the [ark-analysis README](https://github.com/angelolab/ark-analysis/blob/bc6685050dfbef4607874fbbadebd4289251c173/README.md#example-dataset). If you want to use your own data, skip the cell below


In [3]:
#example_dataset.get_example_dataset(dataset="cluster_pixels", save_dir = base_dir, overwrite_existing = False)

## 1: set file paths and parameters

### All data, images, files, etc. must be placed in the 'data' directory, and referenced via '../data/path_to_your_data'


In [4]:
# set up file paths
tiff_dir = os.path.join(base_dir, "image_data")
deepcell_output_dir = os.path.join(base_dir, "segmentation", "deepcell_output")
nimbus_output_dir = os.path.join(base_dir, "nimbus_output")

# Create nimbus output directory
os.makedirs(nimbus_output_dir, exist_ok=True)

# Check if paths exist
io_utils.validate_paths([base_dir, tiff_dir, deepcell_output_dir, nimbus_output_dir])

## 2: Set up input paths and the naming convention for the segmentation data
Store names of channels to exclude in the list below. Either predict all FOVs or specify manually the ones you want to apply Nimbus on.

In [5]:
# define the channels to include
include_channels = [
    "SLIDE-0272_1.0.4_R000_DAPI__FINAL_F",
    "SLIDE-0272_10.0.4_R000_Cy5_CD45-CST-AF647_FINAL_AFR_F",
    "SLIDE-0272_3.0.4_R000_Cy7_Lgals4-750_FINAL_AFR_F",
    "SLIDE-0272_3.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F",
    "SLIDE-0272_9.0.4_R000_Cy7_PanCK-ae1-ae3-750_FINAL_AFR_F",
    "SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555_FINAL_AFR_F",
]

# either get all fovs in the folder...
#fov_names = os.listdir(tiff_dir)
# ... or optionally, select a specific set of fovs manually
fov_names = ['SLIDE-0272']

# make sure to filter paths out that don't lead to FoVs, e.g. .DS_Store files.
fov_names = [fov_name for fov_name in fov_names if not fov_name.startswith(".")] 

# construct paths for fovs
fov_paths = [os.path.join(tiff_dir, fov_name) for fov_name in fov_names]

Define the naming convention for the segmentation data in function `segmentation_naming_convention`, that maps the `fov_name` to the path of the associated segmentation output. The below function `prep_deepcell_naming_convention` assumes that all segmentation outputs are stored in one folder, with the `fov_name` as the prefix and `_whole_cell.tiff` as the suffix, as shown below in the visualization of the folder structure. If this does not apply to your data, you have to define a function `segmentation_naming_convention` that takes an element from `fov_paths` and returns a valid path to the segmentation label map you want to use for that fov.

```
|-- base_dir
|   |-- image_data
|   |   |-- fov_1
|   |   |-- fov_2
|   |-- segmentation
|   |   |-- deepcell_output
|   |   |   |-- fov_1_whole_cell.tiff
|   |   |   |-- fov_2_whole_cell.tiff
|   |-- nimbus_output
```

In [6]:
# Prepare segmentation naming convention that maps a fov_path to the according segmentation label map
segmentation_naming_convention = prep_naming_convention(deepcell_output_dir)

# test segmentation_naming_convention
if os.path.exists(segmentation_naming_convention(fov_paths[0])):
    print("Segmentation data exists for fov 0 and naming convention is correct")
else:
    print("Segmentation data does not exist for fov 0 or naming convention is incorrect")

Segmentation data exists for fov 0 and naming convention is correct


Next we will use the `MultiplexDataset` class to abstract away differences in data representation. The class takes `fov_paths`, `segmentation_naming_convention` and a `suffix` and provides methods `.get_channel(fov, channel)` and `.get_segmentation(fov)` to access the data. The `suffix` is used to filter out files that do not end with the specified suffix. When you use `.ome.tiff` files make sure to set the suffix to `.ome.tiff`, otherwise the ViewerWidget won't be able to display the images.

In [7]:
fov_paths

['/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0272']

In [7]:
dataset = MultiplexDataset(
    fov_paths=fov_paths,
    suffix=".ome.tif", # or .png, .jpg, .jpeg, .tif or .ome.tiff
    include_channels=include_channels,
    segmentation_naming_convention=segmentation_naming_convention,
    output_dir=	nimbus_output_dir,
)

Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0272'].
Channels: ['SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555_FINAL_AFR_F', 'SLIDE-0272_3.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F', 'SLIDE-0272_9.0.4_R000_Cy7_PanCK-ae1-ae3-750_FINAL_AFR_F', 'SLIDE-0272_10.0.4_R000_Cy5_CD45-CST-AF647_FINAL_AFR_F', 'SLIDE-0272_3.0.4_R000_Cy7_Lgals4-750_FINAL_AFR_F', 'SLIDE-0272_1.0.4_R000_DAPI__FINAL_F'].


## 3: Load model and initialize Nimbus application
The following code initializes the Nimbus application and loads the model checkpoint. The model was trained on a diverse set of tissues, protein markers, imaging platforms and cell types and doesn't need re-training. If you want to use the model on a machine without GPU, set `test_time_aug=False` to speed up inference. If you run it on a laptop GPU and run into out-of-memory errors, consider reducing the `batch_size` to 1 and the `input_shape` to `[512,512]`.

In [8]:
nimbus = Nimbus(
    dataset=dataset,
    save_predictions=True,
    batch_size=16,
    test_time_aug=True,
    input_shape=[1024,1024],
    device="auto",
    output_dir=nimbus_output_dir,
    compile_model=True, #used to be False
    mixed_precision=False,
)

# check if all inputs are valid
nimbus.check_inputs()

Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /data1/lowes/ratnayn/conda_envs/nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /data1/lowes/ratnayn/conda_envs/nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Compiling model with torch.compile()...
Model compiled successfully.
All inputs are valid.


## 4: Prepare normalization dictionary 
The next step is to iterate through all the fovs and calculate the 0.999 marker expression quantile for each marker individually. This is used for normalizing the marker expressions prior to predicting marker confidence scores with our model. You can set `n_subset` to estimate the quantiles on a small subset of the data and you can set `multiprocessing=True` to speed up computation.

In [9]:
dataset.prepare_normalization_dict(
    quantile=0.999,
    n_subset=50,
    clip_values=(0, 2),
    multiprocessing=True,
    overwrite=True
)

Iterate over fovs...


## 5: Make predictions with the model
Nimbus will iterate through your samples and store predictions and a file named `nimbus_cell_table.csv` that contains the mean-per-cell predicted marker confidence scores in the sub-directory called `nimbus_output`.

In [ ]:
import time

t0 = time.time()
cell_table = nimbus.predict_fovs()
print(f"predict_fovs took {time.time() - t0:.2f} seconds") #single channel was 450 s

Available GPUs:  1
Predictions will be saved in /data1/lowes/ratnayn/Data/CellDive_analysis_data/nimbus_output
Iterating through fovs will take a while...
Predicting /data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0272...


  0%|          | 0/6 [00:00<?, ?it/s]

Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /data1/lowes/ratnayn/conda_envs/nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /data1/lowes/ratnayn/conda_envs/nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Compiling model with torch.compile()...
Model compiled successfully.


  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

...

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

In [19]:
cell_table

,label,fov,HLADR,GLUT1,ECAD,CD45,Vim,CK17,IDO,Ki67,...,SMA,CD14,PD1,CD3,CD8,CD163,Collagen1,CD68,CD4,CD31
0,1,fov4,0.029052,0.781491,0.464690,0.029485,0.032555,0.027366,0.027724,0.302189,...,0.028725,0.031286,0.028879,0.029522,0.029642,0.031455,0.034914,0.028280,0.031131,0.029025
1,2,fov4,0.041831,0.702823,0.702577,0.098603,0.048976,0.037850,0.039012,0.083631,...,0.150361,0.051384,0.035750,0.044655,0.034452,0.039986,0.135173,0.034420,0.045847,0.306917
2,3,fov4,0.027192,0.575712,0.480484,0.095237,0.029867,0.027168,0.028418,0.758971,...,0.029674,0.110929,0.027166,0.027918,0.030128,0.031862,0.171664,0.041961,0.027497,0.027230
3,4,fov4,0.032616,0.670531,0.756120,0.511150,0.038185,0.029867,0.035788,0.359150,...,0.216104,0.226594,0.031378,0.038585,0.027812,0.664922,0.310459,0.027403,0.266826,0.042293
4,5,fov4,0.031718,0.374812,0.694993,0.032370,0.039562,0.028057,0.029245,0.029727,...,0.029115,0.110616,0.029493,0.026527,0.029572,0.029981,0.033752,0.029824,0.032575,0.031718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15331,1245,fov3,0.051157,0.493232,0.028752,0.774031,0.106251,0.026655,0.032260,0.030265,...,0.027482,0.027999,0.024381,0.783092,0.025551,0.024475,0.065272,0.028017,0.793431,0.029099
15332,1246,fov3,0.052846,0.346239,0.036710,0.532520,0.167573,0.036819,0.035208,0.048123,...,0.032601,0.087240,0.030057,0.417912,0.044539,0.482837,0.074725,0.281505,0.515963,0.041301
15333,1247,fov3,0.588811,0.499337,0.036278,0.552741,0.148397,0.035834,0.532768,0.036725,...,0.033239,0.106775,0.029821,0.186934,0.068026,0.108145,0.131513,0.401519,0.547105,0.074397
15334,1248,fov3,0.356292,0.388547,0.041126,0.432210,0.452110,0.033046,0.317445,0.032938,...,0.114282,0.054396,0.031229,0.382247,0.030281,0.059365,0.370748,0.049272,0.403150,0.033732


## 6: View multiplexed channels and Nimbus predictions side-by-side
Select an FOV and one marker image per channel to inspect the imaging data and associated Nimbus predictions

In [20]:
viewer = NimbusViewer(dataset=dataset, output_dir=nimbus_output_dir)
viewer.display()